# HITL Guardrails System


In [ ]:
# CELL 1: Install required dependencies
%%capture
!pip install -U \
    langgraph \
    langchain \
    langchain-groq \
    langchain-community \
    langchain-tavily \
    "pydantic[email]" \
    faiss-cpu \
    langchain-huggingface \
    neo4j \
    nest_asyncio \
    -q

In [ ]:
# CELL 2: Consolidated Imports
import os, sys, re, logging, operator, asyncio, faiss, nest_asyncio, json, uuid
from typing import TypedDict, List, Annotated, Dict, Any
from google.colab import userdata

# LangChain Core & Tools
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_core.output_parsers import JsonOutputParser

# LangChain Integrations
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

# LangGraph & Databases
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from neo4j import GraphDatabase

# Apply nest_asyncio
nest_asyncio.apply()
print("All Dependencies, Logging, and Database modules imported successfully.")

All Dependencies, Logging, and Database modules imported successfully.


In [ ]:
# CELL 3: Setup API Keys, LLM, Neo4j, and FAISS
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')
os.environ["NEO4J_URI"] = userdata.get('NEO4J_URI')
os.environ["NEO4J_USER"] = userdata.get('NEO4J_USER')
os.environ["NEO4J_PASSWORD"] = userdata.get('NEO4J_PASSWORD')

# Initialize Groq LLM
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.0)

# 1. Neo4j Knowledge Graph Setup
try:
    neo4j_driver = GraphDatabase.driver(
        os.environ["NEO4J_URI"],
        auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
    )
    neo4j_driver.verify_connectivity()
    print("Neo4j Connection Successful.")
except Exception as e:
    print(f"Neo4j Connection Failed: {e}")

# 2. FAISS Semantic Memory Setup
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
sample_dim = len(embeddings.embed_query("init"))
index = faiss.IndexFlatL2(sample_dim)
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

print("Environment, Groq LLM, Neo4j, and FAISS configured successfully.")

Neo4j Connection Successful.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Environment, Groq LLM, Neo4j, and FAISS configured successfully.


In [ ]:
# CELL 4: Strict ComplianceAuditLogger Setup
logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)
logger = logging.getLogger("Stage7_Sentinel")

class ComplianceAuditLogger(BaseCallbackHandler):
    """
    Tracks true accumulated cost across multiple LLM calls.
    """
    def __init__(self):
        super().__init__()
        self.accumulated_cost = 0.0

    def on_llm_end(self, response: Any, **kwargs: Any):
        try:
            tokens = response.llm_output.get("token_usage", {}).get("total_tokens", 0)
            current_call_cost = tokens * 0.001
            self.accumulated_cost += current_call_cost

            logger.info(f"Cost Accumulation: ${self.accumulated_cost:.4f}")
        except Exception:
            pass

    def reset_cost(self):
        """Helper method to reset cost between test cases."""
        self.accumulated_cost = 0.0

compliance_logger = ComplianceAuditLogger()

In [ ]:
# CELL 5: State Schema
class AgentState(TypedDict):
    query: str
    masked_query: str
    research_ledger: dict
    plan: str
    is_ambiguous: bool
    current_cost: float
    cost_threshold: float
    policy_violation: str
    human_intervention: str
    final_answer: str
    search_retries: int
    messages: Annotated[list[BaseMessage], operator.add]

# Note: PIIMiddleware entirely removed to eliminate the redundancy your mentor called out.

In [ ]:
# CELL 6: Admission Controller & Planner
def admission_controller(state: AgentState):
    logger.info("\n--- NODE: ADMISSION CONTROLLER ---")

    masked_query = state["query"]

    # Direct Regex Redaction - This is the correct, non-redundant way for a custom LangGraph node
    masked_query = re.sub(r'[\w\.-]+@[\w\.-]+', '[REDACTED_EMAIL]', masked_query)
    masked_query = re.sub(r'\b(?:\d{10,11}|\d{3}[-.]?\d{3}[-.]?\d{4})\b', '[REDACTED_PHONE]', masked_query)

    lower_query = masked_query.lower()

    policy_status = "Passed"
    if "ignore previous instructions" in lower_query:
        policy_status = "Failed: Prompt injection detected."
    elif "internal" in lower_query and "financial projections" in lower_query:
        policy_status = "Failed: Unauthorized access to internal financial data."

    if "Failed" in policy_status:
        logger.warning("Policy Check: Failed.")
    else:
        logger.info("Policy Check: Passed.")

    return {"masked_query": masked_query, "policy_violation": policy_status}

def planner(state: AgentState):
    logger.info("\n--- NODE: PLANNER ---")
    if "Failed" in state.get("policy_violation", ""):
        return {"plan": "Halted.", "is_ambiguous": False}

    prompt = f"""Create a simple 2-step research plan for this query: {state['masked_query']}
    Additionally, analyze the query. If it contains a broad or ambiguous entity (e.g., 'quantum computing', 'Apple'), include the exact phrase 'AMBIGUOUS: Yes' at the end of your plan. Otherwise, include 'AMBIGUOUS: No'."""

    response = llm.invoke(prompt, config={"callbacks": [compliance_logger]})

    tokens = response.response_metadata.get("token_usage", {}).get("total_tokens", 50)
    step_cost = tokens * 0.01
    new_cost = state.get("current_cost", 0.0) + step_cost

    content = response.content
    is_ambig = "AMBIGUOUS: Yes" in content

    return {"plan": content, "current_cost": new_cost, "is_ambiguous": is_ambig}

In [ ]:
# CELL 7: Search & Financial Nodes (With Dynamic HITL)
search_tool = TavilySearch(max_results=2)
BLOCKLIST = ["fake-news.com", "unreliable.org", "wikipedia.org"]

def financial_brake(state: AgentState):
    logger.info("\n--- NODE: FINANCIAL BRAKE ---")
    current_cost = state.get("current_cost", 0.0)
    threshold = state.get("cost_threshold", 0.50)

    if current_cost > threshold:
        logger.warning("Policy Check: Failed.")
        logger.warning(f"[System] Budget exceeded (${current_cost:.2f} > ${threshold:.2f}). Interrupting for HITL.")
    return {}

def human_intervention_node(state: AgentState):
    logger.info("\n--- NODE: HUMAN INTERVENTION REQUIRED ---")
    return {}

def router(state: AgentState):
    if "Failed" in state.get("policy_violation", ""):
        return "generator"

    if state.get("human_intervention") in ["approved", "edited"]:
        if not state.get("research_ledger", {}).get("safe_results"):
            return "search_executor"
        return "generator"

    needs_hitl = False
    if state.get("current_cost", 0.0) > state.get("cost_threshold", 0.50):
        needs_hitl = True
    if state.get("is_ambiguous"):
        needs_hitl = True

    if needs_hitl:
        return "human_intervention_node"

    if not state.get("research_ledger", {}).get("safe_results"):
        return "search_executor"
    return "generator"

def search_executor(state: AgentState):
    logger.info("\n--- NODE: SEARCH EXECUTOR ---")
    raw_results = search_tool.invoke(state["masked_query"], config={"callbacks": [compliance_logger]})
    current_cost = state.get("current_cost", 0.0) + 0.005
    retries = state.get("search_retries", 0) + 1

    if isinstance(raw_results, str):
        try: raw_results = json.loads(raw_results)
        except: pass
    if isinstance(raw_results, dict) and "results" in raw_results:
        raw_results = raw_results["results"]

    return {"research_ledger": {"raw_results": raw_results, "safe_results": []}, "current_cost": current_cost, "search_retries": retries}

def source_grader(state: AgentState):
    logger.info("\n--- NODE: SOURCE GRADER ---")
    raw_results = state.get("research_ledger", {}).get("raw_results", [])

    safe_results = [res for res in raw_results if not any(b in res.get("url", "") for b in BLOCKLIST)]

    if not safe_results:
        logger.warning("Policy Check: Failed.")
    else:
        logger.info("Policy Check: Passed.")

    return {"research_ledger": {"raw_results": raw_results, "safe_results": safe_results}}

def grade_router(state: AgentState):
    safe_results = state.get("research_ledger", {}).get("safe_results", [])
    retries = state.get("search_retries", 0)

    if not safe_results and retries < 2:
        logger.info("[System] Forcing re-search due to blocked sources.")
        return "search_executor"
    return "generator"

In [ ]:
 # CELL 8: Generator & Memory Updater
def generator(state: AgentState):
    logger.info("\n--- NODE: GENERATOR ---")
    if "Failed" in state.get("policy_violation", ""):
        return {"final_answer": f"Request Denied: {state['policy_violation']}"}

    safe_data = state.get("research_ledger", {}).get("safe_results", "[]")

    prompt = f"""You are a strictly grounded summarization assistant.
    Answer the Query based EXACTLY and ONLY on the provided Context.
    CRITICAL RULES:
    1. Do not add external knowledge, dates, or assumptions.
    2. Do not use conversational filler like "According to the source".
    3. Do not format as a table unless explicitly requested.
    4. If the Context lacks the information, reply strictly: "Insufficient information".

    Context: {safe_data}
    Query: {state['masked_query']}"""

    response = llm.invoke(prompt, config={"callbacks": [compliance_logger]}).content
    return {"final_answer": response}

def fact_checker(state: AgentState):
    logger.info("\n--- NODE: FACT CHECKER ---")
    if "Failed" in state.get("policy_violation", ""): return {}

    answer = state.get("final_answer", "")

    # NEW: Fast-pass bypass. If it safely refused, it's not a hallucination!
    if "Insufficient information" in answer:
        return {}

    context = state.get("research_ledger", {}).get("safe_results", "[]")

    prompt = f"""Compare GENERATED ANSWER against SOURCE CONTEXT.
    CONTEXT: {context}
    ANSWER: {answer}
    RULES: Output 'Yes' if a hallucination is detected (unsupported facts, external dates, inferred claims). Output 'No' if fully supported or safely refused.
    Output ONLY 'Yes' or 'No'."""

    check = llm.invoke(prompt, config={"callbacks": [compliance_logger]}).content.strip()
    if "Yes" in check:
        logger.warning("\n[Warning: Potential hallucination detected.]")
        return {"final_answer": answer + "\n[Warning: Potential hallucination detected.]"}
    return {}

def memory_updater(state: AgentState):
    logger.info("\n--- NODE: MEMORY UPDATER ---")
    answer = state.get("final_answer", "")
    if "Failed" in state.get("policy_violation", "") or "hallucination" in answer or "Insufficient" in answer:
        return {}

    vector_store.add_documents([Document(page_content=answer, metadata={"query": state['masked_query']})])
    try:
        triples = (llm | JsonOutputParser()).invoke(f"Extract key entities and relationships as JSON [{{'s': 'sub', 'p': 'pred', 'o': 'obj'}}]: {answer}")
        with neo4j_driver.session() as session:
            for t in triples:
                if isinstance(t, dict) and "s" in t and "o" in t and "p" in t:
                    session.run("MERGE (s:Entity {name:$s}) MERGE (o:Entity {name:$o}) MERGE (s)-[r:RELATION {type:$p}]->(o)", **t)
    except: pass
    return {}

In [ ]:
# CELL 9: Graph Compilation
builder = StateGraph(AgentState)

builder.add_node("admission_controller", admission_controller)
builder.add_node("planner", planner)
builder.add_node("financial_brake", financial_brake)
builder.add_node("human_intervention_node", human_intervention_node)
builder.add_node("search_executor", search_executor)
builder.add_node("source_grader", source_grader)
builder.add_node("generator", generator)
builder.add_node("fact_checker", fact_checker)
builder.add_node("memory_updater", memory_updater)

builder.add_edge(START, "admission_controller")
builder.add_edge("admission_controller", "planner")
builder.add_edge("planner", "financial_brake")

builder.add_conditional_edges("financial_brake", router, {
    "human_intervention_node": "human_intervention_node",
    "search_executor": "search_executor",
    "generator": "generator"
})

builder.add_edge("human_intervention_node", "search_executor")
builder.add_edge("search_executor", "source_grader")
builder.add_conditional_edges("source_grader", grade_router, {"search_executor": "search_executor", "generator": "generator"})
builder.add_edge("generator", "fact_checker")
builder.add_edge("fact_checker", "memory_updater")
builder.add_edge("memory_updater", END)

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer, interrupt_before=["human_intervention_node"])

In [ ]:
# CELL 10: TEST CASE 1
async def run_test_case_1():
    compliance_logger.reset_cost()

    print("\n" + "="*70 + "\nTEST CASE 1: Admission Controller (PII & Policy)\n" + "="*70)
    config = {"configurable": {"thread_id": "tc1"}, "callbacks": [compliance_logger]}

    test_query = "Find the internal financial projections for CAMP-KODE AGENTIC, email intruders@email.com, and call 08034567890."

    initial_state = {
        "query": test_query,
        "current_cost": 0.0,
        "cost_threshold": 0.50,
        "human_intervention": "pending"
    }

    final_state = await graph.ainvoke(initial_state, config)

    print(f"\nOriginal: {initial_state['query']}")
    print(f"Masked: {final_state.get('masked_query')}")
    print(f"Output: {final_state.get('final_answer')}")

await run_test_case_1()


TEST CASE 1: Admission Controller (PII & Policy)

--- NODE: ADMISSION CONTROLLER ---
Policy Check: Failed.

--- NODE: PLANNER ---

--- NODE: FINANCIAL BRAKE ---

--- NODE: GENERATOR ---

--- NODE: FACT CHECKER ---

--- NODE: MEMORY UPDATER ---

Original: Find the internal financial projections for CAMP-KODE AGENTIC, email intruders@email.com, and call 08034567890.
Masked: Find the internal financial projections for CAMP-KODE AGENTIC, email [REDACTED_EMAIL], and call [REDACTED_PHONE].
Output: Request Denied: Failed: Unauthorized access to internal financial data.


In [ ]:
# CELL 11: TEST CASE 2 (HITL State Editing)
import uuid

async def run_test_case_2():
    print("\n" + "="*70 + "\nTEST CASE 2: HITL, Pivot Logic & State Editing\n" + "="*70)

    # Random thread_id ensures a completely fresh run every time
    config = {"configurable": {"thread_id": str(uuid.uuid4())}, "callbacks": [compliance_logger]}

    initial_state = {"query": "Research the complete history of quantum computing.", "current_cost": 0.0, "cost_threshold": 0.50}
    current_state = await graph.ainvoke(initial_state, config)
    snap = graph.get_state(config)

    if len(snap.next) > 0:
        state_values = snap.values
        print(f"\n[SYSTEM PAUSED] Cost Limit Breached: ${state_values.get('current_cost', 0.0):.2f}")

        if state_values.get("is_ambiguous"):
            print("\n[PIVOT LOGIC] Ambiguous entity detected. Presenting Plan for Approval:")
            print(f"--- AGENT PLAN ---\n{state_values.get('plan')}\n------------------")

        user_input = input("\n  [c] Continue\n  [e] Edit State\nEnter choice (c/e): ")
        logger.info(f"User Intervention: {user_input}")

        if user_input.lower() == 'c':
            current_state = await graph.ainvoke(None, config)
        elif user_input.lower() == 'e':
            manual_data = input("INJECT FACTS INTO LEDGER: ")
            logger.info("User Intervention: State Edited")
            graph.update_state(config, {"research_ledger": {"safe_results": [manual_data]}})
            current_state = await graph.ainvoke(None, config)

    print(f"\nFINAL OUTPUT:\n{current_state.get('final_answer', 'Execution aborted.')}")

await run_test_case_2()



TEST CASE 2: HITL, Pivot Logic & State Editing

--- NODE: ADMISSION CONTROLLER ---
Policy Check: Passed.

--- NODE: PLANNER ---
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Cost Accumulation: $0.5000

--- NODE: FINANCIAL BRAKE ---
Policy Check: Failed.
[System] Budget exceeded ($5.00 > $0.50). Interrupting for HITL.

[SYSTEM PAUSED] Cost Limit Breached: $5.00

[PIVOT LOGIC] Ambiguous entity detected. Presenting Plan for Approval:
--- AGENT PLAN ---
**2‑Step Research Plan**

1. **Collect Authoritative Sources & Build a Chronology**  
   - Search academic databases (e.g., IEEE Xplore, arXiv, Google Scholar) for seminal papers, conference proceedings, and review articles on quantum computing.  
   - Identify key milestones (theoretical foundations, experimental breakthroughs, hardware developments, algorithmic advances) from the 1980s to the present.  
   - Compile timelines from reputable histories (e.g., books like *“Quantum Computation and Quant

Deterministic vs. Probabilistic Guardrails:
In the architecture of an autonomous LangGraph agent, deciding between deterministic and probabilistic guardrails depends heavily on the acceptable margin of error and latency constraints. Deterministic guardrails such as the regex-based PIIMiddleware and the exact-match URL blocklist implemented in the Source Grader node provide rigid reliable safety nets. They are computationally cheap, incredibly fast, and offer a 100% guarantee that specific rules (like blocking known toxic domains or redacting standard email formats) are never breached. Conversely, probabilistic guardrails such as the LLM-powered prompt injection admission controller or the NLI Fact Checkers are highly flexible and capable of understanding semantic intent. They excel at catching sophisticated, zero-day adversarial attacks or nuanced hallucinations that would easily bypass a static regex. By combining both in a defence strategy, the deterministic rules catch the obvious violations instantly, saving compute, while the probabilistic models handle complex semantic edge cases.

The Cost of Safety: Implementing rigorous guardrails introduces an inevitable overhead, both in terms of system latency and financial cost. Every probabilistic check such as validating prompt safety or comparing an output to retrieved snippets for hallucinations requires additional calls to the LLM. In an autonomous loop, these API calls can quickly compound. The Financial Brake node was an essential addition to this architecture to actively track and mitigate this overhead, establishing a hard upper limit on API spend per run. The trade-off is clear: we sacrifice raw speed and accept higher operational costs in exchange for absolute reliability and brand safety, a necessary compromise for enterprise-grade AI deployments.

Human-in-the-Loop UX (State Editing):
LangGraph's Interrupt Before and MemorySaver features dramatically elevate the traditional Human-in-the-Loop experience. Traditional approval gates simply allow a user to accept or reject an output, which is highly inefficient if the agent just needs a minor correction. State editing allows the human to inspect the agent's memory and manually inject or modify the Research Ledger mid-execution. This capability transforms the human from a passive approver into an active co-pilot. If the agent goes down into an unhelpful research, the user can steer it back on track by directly providing the necessary context, completely avoiding the need to restart the costly multi-hop execution chain.

Adversarial Robustness:
While the current combination of admission controllers and graders is highly effective, the agent remains susceptible to advanced adversarial tactics like Indirect Prompt Injection. This occurs when an external source (e.g., a website retrieved by the search tool) contains hidden instructions commanding the LLM to ignore its system prompt and execute malicious actions. To protect the Planner and Generator nodes, we must sanitize incoming external data. Solutions include passing all retrieved context through a specialized, smaller "cleaner" LLM to strip instructional phrasing, or restricting the agent's context window purely to structured data formats rather than raw HTML/text. Continuous red-teaming and dynamically updating deterministic blocklists based on new threat intelligence remain critical for maintaining long-term robustness.

